In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score, accuracy_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

In [2]:
# Load the dataset
df = pd.read_csv('course_lead_scoring_2026.csv')

# Check the first few rows
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [3]:
df.shape

(5000, 9)

In [4]:
df.dtypes

lead_source                  object
industry                     object
employment_status            object
location                     object
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [5]:
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [6]:
# Identify categorical and numerical columns

categorical = list(df.select_dtypes(include=['object']).columns)
numerical = list(df.select_dtypes(exclude=['object']).columns)

categorical, numerical

(['lead_source', 'industry', 'employment_status', 'location'],
 ['annual_income',
  'number_of_courses_viewed',
  'interaction_count',
  'lead_score',
  'converted'])

In [7]:
if 'converted' in numerical:
    numerical.remove('converted')

In [8]:
# Categorical missing values -> 'NA'
df[categorical] = df[categorical].fillna('NA')

# Numerical missing values -> 0.0
df[numerical] = df[numerical].fillna(0.0)

In [9]:
df.isnull().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

In [11]:
# 1)
df['industry'].mode()

0    technology
Name: industry, dtype: object

In [12]:
# Numerical features only
df[numerical].corr()

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000000,0.161300,0.122842,0.229496
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204
interaction_count,0.122842,0.721609,1.000000,0.915746
lead_score,0.229496,0.757204,0.915746,1.000000


In [17]:
# 2)
pairs = [
    ('interaction_count', 'lead_score'),
    ('number_of_courses_viewed', 'lead_score'),
    ('number_of_courses_viewed', 'interaction_count'),
    ('annual_income', 'interaction_count')
]

for col1, col2 in pairs:
    corr = df[col1].corr(df[col2])
    print(f'{col1} and {col2}: {corr:.4f}')

interaction_count and lead_score: 0.9157
number_of_courses_viewed and lead_score: 0.7572
number_of_courses_viewed and interaction_count: 0.7216
annual_income and interaction_count: 0.1228


In [18]:
df_full_train, df_test = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

df_train, df_val = train_test_split(
    df_full_train,
    test_size=0.25,
    random_state=42
)

In [19]:
len(df_train), len(df_val), len(df_test)

(3000, 1000, 1000)

In [20]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [21]:
y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

In [22]:
del df_train['converted']
del df_val['converted']
del df_test['converted']

In [23]:
df_train.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score
0,paid_ads,finance,employed,asia,65980.0,2,3,0.37
1,events,finance,employed,south_america,71662.0,0,0,0.09
2,referral,NA,employed,asia,40220.0,2,5,0.55
3,organic_search,finance,employed,north_america,88352.0,2,2,0.45
4,events,technology,employed,north_america,89316.0,1,5,0.00


In [24]:
def mutual_info_converted_score(series):
    return mutual_info_score(series, y_train)

In [25]:
mi = df_train[categorical].apply(mutual_info_converted_score)
mi = mi.sort_values(ascending=False)

mi

lead_source          0.034346
employment_status    0.018966
industry             0.001995
location             0.001043
dtype: float64

In [26]:
# 3)
mi.round(2)

lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64

In [28]:
categorical

['lead_source', 'industry', 'employment_status', 'location']

In [29]:
numerical

['annual_income',
 'number_of_courses_viewed',
 'interaction_count',
 'lead_score']

In [30]:
train_dicts = df_train[categorical + numerical].to_dict(orient='records')

In [31]:
dv = DictVectorizer(sparse=False)

X_train = dv.fit_transform(train_dicts)

In [32]:
X_train.shape

(3000, 28)

In [33]:
model = LogisticRegression(
    solver='liblinear',
    C=1.0,
    max_iter=1000,
    random_state=42
)

model.fit(X_train, y_train)

LogisticRegression(max_iter=1000, random_state=42, solver='liblinear')

In [34]:
val_dicts = df_val[categorical + numerical].to_dict(orient='records')

X_val = dv.transform(val_dicts)

In [35]:
y_pred = model.predict_proba(X_val)[:, 1]

In [36]:
converted_decision = (y_pred >= 0.5)

In [40]:
# 4)

accuracy = (y_val == converted_decision).mean()

accuracy

np.float64(0.645)

In [42]:
def train_without_feature(feature_to_remove):
    
    features = categorical + numerical
    features = [f for f in features if f != feature_to_remove]
    
    train_dicts = df_train[features].to_dict(orient='records')
    val_dicts = df_val[features].to_dict(orient='records')
    
    dv_temp = DictVectorizer(sparse=False)
    
    X_train_temp = dv_temp.fit_transform(train_dicts)
    X_val_temp = dv_temp.transform(val_dicts)
    
    model_temp = LogisticRegression(
        solver='liblinear',
        C=1.0,
        max_iter=1000,
        random_state=42
    )
    
    model_temp.fit(X_train_temp, y_train)
    
    y_pred_temp = model_temp.predict_proba(X_val_temp)[:, 1]
    predictions = y_pred_temp >= 0.5
    
    accuracy_temp = (y_val == predictions).mean()
    
    return accuracy_temp

In [43]:
features_to_test = [
    'lead_source',
    'number_of_courses_viewed',
    'interaction_count'
]

In [44]:
results = []

for feature in features_to_test:
    
    acc = train_without_feature(feature)
    difference = original_accuracy - acc
    
    results.append({
        'feature_removed': feature,
        'accuracy': acc,
        'difference': difference
    })

results_df = pd.DataFrame(results)

results_df

,feature_removed,accuracy,difference
0,lead_source,0.642,0.003
1,number_of_courses_viewed,0.643,0.002
2,interaction_count,0.601,0.044


In [45]:
# 5)
results_df.sort_values('difference')

,feature_removed,accuracy,difference
1,number_of_courses_viewed,0.643,0.002
0,lead_source,0.642,0.003
2,interaction_count,0.601,0.044


In [46]:
C_values = [
    0.000001,
    0.00001,
    0.0001,
    0.001
]

In [47]:
results = []

for C in C_values:
    
    model = LogisticRegression(
        solver='liblinear',
        C=C,
        max_iter=1000,
        random_state=42
    )
    
    model.fit(X_train, y_train)
    
    y_pred = model.predict_proba(X_val)[:, 1]
    predictions = y_pred >= 0.5
    
    accuracy = (y_val == predictions).mean()
    
    results.append({
        'C': C,
        'accuracy': accuracy,
        'rounded_accuracy': round(accuracy, 3)
    })

results_df = pd.DataFrame(results)

results_df

,C,accuracy,rounded_accuracy
0,0.000001,0.598,0.598
1,0.000010,0.598,0.598
2,0.000100,0.613,0.613
3,0.001000,0.645,0.645


In [48]:
results_df.sort_values(
    ['rounded_accuracy', 'C'],
    ascending=[False, True]
)

,C,accuracy,rounded_accuracy
3,0.001000,0.645,0.645
2,0.000100,0.613,0.613
0,0.000001,0.598,0.598
1,0.000010,0.598,0.598
